In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

In [3]:
# --------------------------------------------------
# 1. Dataset
# --------------------------------------------------

data = [
    ("i am a student", "je suis etudiant"),
    ("i like cats", "j aime les chats"),
    ("i love dogs", "j aime les chiens"),
    ("i am happy", "je suis heureux"),
    ("i like music", "j aime la musique"),
]

In [74]:
# --------------------------------------------------
# 2. Build vocabulary
# --------------------------------------------------

def build_vocab(sentences):

    vocab = {
        "<pad>": 0,
        "<sos>": 1,
        "<eos>": 2,
        "<unk>": 3
    }

    for sentence in sentences:

        for word in sentence.split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

src_vocab = build_vocab([x[0] for x in data])
trg_vocab = build_vocab(x[1] for x in data)

print(src_vocab)
print(trg_vocab)

{'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3, 'i': 4, 'am': 5, 'a': 6, 'student': 7, 'like': 8, 'cats': 9, 'love': 10, 'dogs': 11, 'happy': 12, 'music': 13}
{'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3, 'je': 4, 'suis': 5, 'etudiant': 6, 'j': 7, 'aime': 8, 'les': 9, 'chats': 10, 'chiens': 11, 'heureux': 12, 'la': 13, 'musique': 14}


In [75]:
def numericalize(sentence, vocab):
    token = ["<sos>"] + sentence.split() + ["<eos>"]
    return torch.tensor(
        [vocab.get(word, vocab["<unk>"]) for word in token],
        dtype=torch.long
    )

src_seq = [numericalize(sentence, src_vocab) for sentence, _ in data]
trg_seq = [numericalize(sentence, trg_vocab) for _, sentence in data]

print(src_seq)
print(trg_seq)

[tensor([1, 4, 5, 6, 7, 2]), tensor([1, 4, 8, 9, 2]), tensor([ 1,  4, 10, 11,  2]), tensor([ 1,  4,  5, 12,  2]), tensor([ 1,  4,  8, 13,  2])]
[tensor([1, 4, 5, 6, 2]), tensor([ 1,  7,  8,  9, 10,  2]), tensor([ 1,  7,  8,  9, 11,  2]), tensor([ 1,  4,  5, 12,  2]), tensor([ 1,  7,  8, 13, 14,  2])]


In [76]:
from torch.nn.utils.rnn import pad_sequence

src = pad_sequence(
    src_seq, batch_first=True, padding_value=src_vocab['<pad>']
)
trg = pad_sequence(
    trg_seq, batch_first=True, padding_value=trg_vocab['<pad>']
)

print(src.shape, trg.shape)
print(src, trg)

torch.Size([5, 6]) torch.Size([5, 6])
tensor([[ 1,  4,  5,  6,  7,  2],
        [ 1,  4,  8,  9,  2,  0],
        [ 1,  4, 10, 11,  2,  0],
        [ 1,  4,  5, 12,  2,  0],
        [ 1,  4,  8, 13,  2,  0]]) tensor([[ 1,  4,  5,  6,  2,  0],
        [ 1,  7,  8,  9, 10,  2],
        [ 1,  7,  8,  9, 11,  2],
        [ 1,  4,  5, 12,  2,  0],
        [ 1,  7,  8, 13, 14,  2]])


In [77]:
class Encoder(nn.Module):

    def __init__(self, input_dim, embedding_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(
            input_dim,
            embedding_dim
        )

        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

    def forward(self, x):
        embedding = self.embedding(x)
        _, hidden = self.gru(embedding)

        return hidden

In [78]:
class Decoder(nn.Module):
    def __init__(self, output_dim, embedding_dim, hidden_dim):
        super().__init__()

        self.output_dim = output_dim
        self.embedding = nn.Embedding(
            output_dim,
            embedding_dim
        )

        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_dim,
            output_dim
        )

    def forward(self, input, hidden):
        input = input.unsqueeze(1)

        embedded = self.embedding(input)
        output, hidden = self.gru(embedded, hidden)

        output = output.squeeze(1)

        prediction = self.fc(output)

        return prediction, hidden

In [79]:
class Seq2Seq(nn.Module):

    def __init__(self, encoder, decoder):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing_ratio=0.5):

        batch_size = src.size(0)

        trg_len = trg.size(1)

        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(
            batch_size,
            trg_len,
            trg_vocab_size
        )

        hidden = self.encoder(src)

        input = trg[:, 0]

        for t in range(1, trg_len):

            output, hidden = self.decoder(
                input,
                hidden
            )

            outputs[:, t] = output

            teacher_force = (random.random() < teacher_forcing_ratio)

            top1 = output.argmax(1)

            input = (
                trg[:, t] if teacher_force else top1)

        return outputs

In [80]:
INPUT_DIM = len(src_vocab)
OUTPUT_DIM = len(trg_vocab)

EMBEDDING_DIM = 64
HIDDEN_DIM = 128

encoder = Encoder(
    INPUT_DIM,
    EMBEDDING_DIM,
    HIDDEN_DIM
)

decoder = Decoder(
    OUTPUT_DIM,
    EMBEDDING_DIM,
    HIDDEN_DIM
)

model = Seq2Seq(
    encoder,
    decoder
)

In [81]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.01
)

criterion = nn.CrossEntropyLoss(
    ignore_index=trg_vocab["<pad>"]
)

In [82]:
for epoch in range(10):

    model.train()

    optimizer.zero_grad()

    output = model(
        src,
        trg,
        teacher_forcing_ratio=0.5
    )

    # Remove <sos>
    output = output[:, 1:, :]

    trg_real = trg[:, 1:]

    output_dim = output.size(-1)

    output = output.reshape(
        -1,
        output_dim
    )

    trg_real = trg_real.reshape(-1)

    loss = criterion(
        output,
        trg_real
    )

    loss.backward()

    optimizer.step()

    print(f"Epoch: {epoch} Loss: {loss.item():.4f}")

Epoch: 0 Loss: 2.7332
Epoch: 1 Loss: 2.2853
Epoch: 2 Loss: 1.6301
Epoch: 3 Loss: 1.2686
Epoch: 4 Loss: 0.8106
Epoch: 5 Loss: 0.6195
Epoch: 6 Loss: 0.3503
Epoch: 7 Loss: 0.3697
Epoch: 8 Loss: 0.2857
Epoch: 9 Loss: 0.2503


In [83]:
def translate_sentence(sentence, model, src_vocab, trg_vocab, max_len=20):

    model.eval()

    # Convert source sentence to token IDs
    tokens = sentence.lower().split()

    tokens = ["<sos>"] + tokens + ["<eos>"]

    src_indices = [
        src_vocab.get(token, src_vocab["<unk>"])
        for token in tokens
    ]

    src_tensor = torch.tensor(
        src_indices,
        dtype=torch.long
    ).unsqueeze(0)

    # Encode source sentence
    with torch.no_grad():
        hidden = model.encoder(src_tensor)

    # Start decoder with <sos>
    trg_indices = [
        trg_vocab["<sos>"]
    ]

    input_token = torch.tensor(
        [trg_vocab["<sos>"]],
        dtype=torch.long
    )

    # Generate tokens one at a time
    with torch.no_grad():

        for _ in range(max_len):

            output, hidden = model.decoder(
                input_token,
                hidden
            )

            # Get most likely token
            predicted_token = output.argmax(1).item()

            trg_indices.append(predicted_token)

            # Stop when <eos> is predicted
            if predicted_token == trg_vocab["<eos>"]:
                break

            # Feed prediction back into decoder
            input_token = torch.tensor(
                [predicted_token],
                dtype=torch.long
            )

    # Convert IDs back to words
    id_to_token = {
        index: token
        for token, index in trg_vocab.items()
    }

    translated_tokens = [
        id_to_token[index]
        for index in trg_indices
    ]

    return translated_tokens

In [85]:
result = translate_sentence(
    "Me",
    model,
    src_vocab,
    trg_vocab
)

print(result)

['<sos>', 'je', 'suis', 'heureux', '<eos>']
